# 05 — Does environmental context improve speed estimates?

This notebook turns the project purpose into a model comparison. The intended user is a fleet operations or voyage performance analyst reviewing movements after a voyage. The output is a context-aware reference that can help prioritize segments for human review. It is not a speed recommendation and does not measure fuel or emissions.

**Main question:** Do weather and ocean measurements improve estimates of observed speed over ground after vessel category, location, time, and local traffic are taken into account?

**Null hypothesis:** Adding environmental variables does not improve predictions on reports from vessels kept out of model fitting.

**Alternative hypothesis:** Environmental variables reduce prediction error by a repeatable amount.

The comparison uses a context-only baseline and the same regression with environmental features added. Reports from a vessel stay in the same training or test group. A second summary checks whether extreme speeds change the conclusion.



## 1. Load the joined sample and identify the available fields

The preceding notebook creates a reproducible 50,000-row position sample and attaches environmental values, match distances, and time-coverage flags. The analysis sample is small enough for a clear first model while the traffic feature itself is calculated from the full AIS position file.


In [1]:
# Import project path tools for locating the joined sample and source positions.
from pathlib import Path
# Import SQLite for a memory-conscious distinct-vessel traffic count.
import sqlite3
# Import temporary-directory support for the intermediate traffic database.
import tempfile
# Import NumPy and pandas for feature preparation and summaries.
import numpy as np
import pandas as pd
# Import PyArrow so AIS positions can be read in manageable batches.
import pyarrow.parquet as pq
# Find the project folder whether Jupyter starts in the root or notebooks folder.
PROJECT = Path.cwd()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
# Point to the folder that stores reproducible derived data.
PROCESSED = PROJECT / 'data' / 'processed'
# Load the matched sample saved by Notebook 04.
joined = pd.read_parquet(PROCESSED / 'joined_environment_sample.parquet')
# Show row count and field names before adding vessel and traffic context.
print(f'Joined position sample: {len(joined):,} rows')
print('Available fields:', ', '.join(joined.columns))
# Show missingness in speed, course, and the matched environmental fields.
display(joined[['sog_knots_clean', 'cog_deg_clean', 'era5_u10', 'era5_v10', 'phy_uo', 'phy_vo', 'wave_VHM0_m']].isna().mean().rename('missing_share').to_frame())


Joined position sample: 50,000 rows
Available fields: Mmsi, Date_utc, Latitude, Longitude, sog_knots_clean, sog_message27_unavailable_flag, cog_deg_clean, NavigationStatus, era5_sp, era5_u10, era5_v10, era5_distance_km, era5_within_time, phy_thetao, phy_uo, phy_vo, phy_distance_km, phy_within_time, wave_VHM0_m, wave_distance_km, wave_within_time, wind_speed_ms, current_speed_ms, all_time_matches


,missing_share
sog_knots_clean,0.00430
cog_deg_clean,0.22042
era5_u10,0.00000
era5_v10,0.00000
phy_uo,0.00000
phy_vo,0.00000
wave_VHM0_m,0.00000


## 2. Attach the latest vessel type reported before each position

The type is looked up by MMSI and report time. A type message older than 30 days is treated as unknown for the primary model, and the linked report age is retained. Thirty days is a working rule for this model, not a universal AIS standard; the summary below shows how many rows it affects.


In [2]:
# Load the vessel ID, report time, ship-type code, and decoded label from cleaned AIS_SPEC.
spec = pd.read_parquet(PROCESSED / 'ais_specifications_clean.parquet', columns=['Mmsi', 'Date_utc', 'ShipType', 'ship_type_label'])
# Rename the vessel-information timestamp so its source is clear after joining.
spec = spec.rename(columns={'Date_utc': 'spec_time_utc', 'ShipType': 'spec_type_code', 'ship_type_label': 'spec_type_label'})
# Sort position and vessel reports by their timestamps for a backward as-of join.
left = joined.sort_values('Date_utc').copy()
right = spec.sort_values('spec_time_utc').copy()
# For each vessel position, attach the latest specification report sent no later than that position.
model_data = pd.merge_asof(left, right, left_on='Date_utc', right_on='spec_time_utc', by='Mmsi', direction='backward')
# Measure the age of the attached vessel-information report in days.
model_data['spec_age_days'] = (model_data['Date_utc'] - model_data['spec_time_utc']).dt.total_seconds() / 86400
# Use type labels only when they are present, nonzero, and recent enough for this analysis.
recent_type = model_data['spec_age_days'].le(30) & model_data['spec_type_code'].notna() & model_data['spec_type_code'].ne(0)
model_data['vessel_type_model'] = model_data['spec_type_label'].where(recent_type, 'Unknown or old report')
# Summarize type availability and report-age handling before modeling.
print('Rows with a type report no older than 30 days:', f'{int(recent_type.sum()):,}', f'({recent_type.mean():.1%})')
print('Rows with no earlier type report:', f'{model_data["spec_time_utc"].isna().sum():,}')
# Display linked-report age quantiles for all available as-of matches.
display(model_data['spec_age_days'].describe(percentiles=[.5, .9, .95, .99]).to_frame('spec_age_days'))
# Show the largest broad vessel groups after handling unavailable and old codes.
display(model_data['vessel_type_model'].value_counts(dropna=False).head(12).to_frame('sample_rows'))
# Load the separate high-speed, nearly stationary-position review list.
high_sog_review = pd.read_csv(PROCESSED / 'high_sog_stationary_position_candidates.csv')
# Convert review timestamps to UTC-aware values to match the analysis timestamps exactly.
high_sog_review['Date_utc'] = pd.to_datetime(high_sog_review['Date'], utc=True)
# Build timestamp and MMSI keys for reports flagged by the track screen.
review_keys = set(zip(high_sog_review['Mmsi'].astype('int64'), high_sog_review['Date_utc']))
# Mark only the exact candidate reports; the cleaned AIS file remains unchanged.
model_keys = zip(model_data['Mmsi'].astype('int64'), model_data['Date_utc'])
model_data['high_sog_stationary_review_flag'] = [key in review_keys for key in model_keys]
# Show how many reports and vessels are under review before defining the model sample.
print(f"High-SOG stationary-position review list: {len(high_sog_review):,} reports across {high_sog_review['Mmsi'].nunique():,} vessels")


Rows with a type report no older than 30 days: 48,974 (97.9%)
Rows with no earlier type report: 259


,spec_age_days
count,49741.000000
mean,0.106362
std,1.455338
min,0.000000
50%,0.008218
90%,0.036470
95%,0.075833
99%,1.062058
max,78.718333


,sample_rows
Pleasure Craft/Sailing,23903
Cargo,6724
Fishing,5363
Other,5308
Tanker,3434
Passenger,2180
Tug Tow,2062
Unknown or old report,1026


High-SOG stationary-position review list: 16 reports across 4 vessels


## 3. Build a local traffic measure from the full AIS file

For each hour and 0.1-degree location cell, count distinct MMSIs observed in that cell. This is a local traffic proxy: it does not describe lane capacity, vessel separation, or navigational risk. The AIS file is scanned in batches, and a small temporary SQLite table prevents the full source from being loaded into memory at once. A vessel's own observation is subtracted when the count is attached to its position.


In [3]:
# Point to the cleaned AIS position file and retain only fields needed for traffic counts.
ais_path = PROCESSED / 'ais_positions_clean.parquet'
# Create traffic keys for one hour and 0.1-degree latitude/longitude cells.
CELL_DEGREES = 0.1
# Convert one hour in nanoseconds to a whole-number time key.
NANOSECONDS_PER_HOUR = 3_600_000_000_000
# Create a temporary folder so the large intermediate database is removed automatically.
with tempfile.TemporaryDirectory(prefix='mmdec_traffic_') as temp_dir:
    # Keep the temporary distinct-vessel table outside the project deliverables.
    database_path = Path(temp_dir) / 'traffic.sqlite'
    # Open the temporary database for incremental inserts.
    connection = sqlite3.connect(database_path)
    # Store unique vessel/hour/location combinations; the primary key prevents double-counting.
    connection.execute('CREATE TABLE vessel_cell_hour (mmsi INTEGER, hour_id INTEGER, lat_bin INTEGER, lon_bin INTEGER, PRIMARY KEY (mmsi, hour_id, lat_bin, lon_bin)) WITHOUT ROWID')
    # Read only the four source columns needed to locate each AIS event.
    parquet_file = pq.ParquetFile(ais_path)
    # Record the number of batches so progress can be shown while the full file is scanned.
    batch_count = (parquet_file.metadata.num_rows + 249_999) // 250_000
    # Process the large position table in bounded batches.
    for batch_number, batch in enumerate(parquet_file.iter_batches(batch_size=250_000, columns=['Mmsi', 'Date', 'Latitude', 'Longitude']), start=1):
        # Convert the current Arrow batch to a temporary pandas frame.
        part = batch.to_pandas()
        # Keep only reports with nonmissing IDs, times, and valid geographic coordinates.
        valid = part['Mmsi'].notna() & part['Date'].notna() & part['Latitude'].between(-90, 90) & part['Longitude'].between(-180, 180)
        part = part.loc[valid, ['Mmsi', 'Date', 'Latitude', 'Longitude']].copy()
        # Convert each report time to an integer hour key using the project's UTC convention.
        part['hour_id'] = part['Date'].astype('int64') // NANOSECONDS_PER_HOUR
        # Assign each position to its 0.1-degree latitude cell.
        part['lat_bin'] = np.floor(part['Latitude'] / CELL_DEGREES).astype('int32')
        # Assign each position to its 0.1-degree longitude cell.
        part['lon_bin'] = np.floor(part['Longitude'] / CELL_DEGREES).astype('int32')
        # Keep only integer keys and remove repeated keys within this batch before inserting.
        keys = part[['Mmsi', 'hour_id', 'lat_bin', 'lon_bin']].drop_duplicates()
        # Insert unique vessel/hour/location combinations; keys repeated in other batches are ignored.
        connection.executemany('INSERT OR IGNORE INTO vessel_cell_hour VALUES (?, ?, ?, ?)', keys.itertuples(index=False, name=None))
        # Commit each batch so memory remains bounded and progress is durable.
        connection.commit()
        # Report progress every tenth batch and on the final batch.
        if batch_number % 10 == 0 or batch_number == batch_count:
            print(f'Processed AIS traffic batch {batch_number:,} of {batch_count:,}')
    # Aggregate distinct vessels by the same hour and location cell.
    traffic = pd.read_sql_query('SELECT hour_id, lat_bin, lon_bin, COUNT(*) AS vessels_in_cell_hour FROM vessel_cell_hour GROUP BY hour_id, lat_bin, lon_bin', connection)
    # Close the temporary database before its temporary folder is cleaned up.
    connection.close()
# Show the size and range of the derived traffic table.
print(f'Traffic cells: {len(traffic):,}; vessels per cell-hour: {traffic["vessels_in_cell_hour"].min()} to {traffic["vessels_in_cell_hour"].max()}')


Processed AIS traffic batch 10 of 76
Processed AIS traffic batch 20 of 76
Processed AIS traffic batch 30 of 76
Processed AIS traffic batch 40 of 76
Processed AIS traffic batch 50 of 76
Processed AIS traffic batch 60 of 76
Processed AIS traffic batch 70 of 76
Processed AIS traffic batch 76 of 76
Traffic cells: 1,621,702; vessels per cell-hour: 1 to 422


In [4]:
# Assign each sampled AIS position to the same 0.1-degree cells used for traffic counts.
model_data['hour_id'] = model_data['Date_utc'].astype('int64') // NANOSECONDS_PER_HOUR
model_data['lat_bin_traffic'] = np.floor(model_data['Latitude'] / CELL_DEGREES).astype('int32')
model_data['lon_bin_traffic'] = np.floor(model_data['Longitude'] / CELL_DEGREES).astype('int32')
# Attach the full-data distinct-vessel count for each matching cell and hour.
model_data = model_data.merge(traffic, left_on=['hour_id', 'lat_bin_traffic', 'lon_bin_traffic'], right_on=['hour_id', 'lat_bin', 'lon_bin'], how='left', validate='many_to_one')
# Treat an empty traffic cell as zero and remove the focal vessel from its own context count.
model_data['other_vessels_cell_hour'] = (model_data['vessels_in_cell_hour'].fillna(0) - 1).clip(lower=0)
# Use a logarithm because traffic counts are usually right-skewed.
model_data['log_other_vessels'] = np.log1p(model_data['other_vessels_cell_hour'])
# Check that sample positions found their source vessel in the full AIS traffic count.
display(model_data[['vessels_in_cell_hour', 'other_vessels_cell_hour']].describe(percentiles=[.5, .9, .95, .99]))


,vessels_in_cell_hour,other_vessels_cell_hour
count,50000.000000,50000.000000
mean,31.279220,30.279220
std,35.352396,35.352396
min,1.000000,0.000000
50%,19.000000,18.000000
90%,75.000000,74.000000
95%,107.000000,106.000000
99%,156.000000,155.000000
max,422.000000,421.000000


'## 4. Set up the comparison sample and relative environmental features

The model uses reports above 0.5 knots as a practical underway screen. All models use the same complete-case sample at a 20 km maximum distance for each environmental grid. The earlier notebook reports coverage sensitivity over several distance limits. A targeted review list marks reports with repeated high speed at nearly fixed positions; only exact matches to those review records are excluded from the primary fit. Other fast vessel reports are kept when present.

Wind and current components are projected along and across the vessel's course. A positive along-course component points in the vessel's direction of travel; a negative component points against it. This is a simple context feature, not a complete hydrodynamic model.

In [5]:
# Keep rows with environmental values from all three datasets inside the time tolerances.
model_data['all_time_matches'] = model_data['era5_within_time'] & model_data['phy_within_time'] & model_data['wave_within_time']
# Apply the provisional 20 km maximum distance to every environmental grid.
model_data['within_20_km'] = model_data['era5_distance_km'].le(20) & model_data['phy_distance_km'].le(20) & model_data['wave_distance_km'].le(20)
# Convert course over ground to radians for along-course and cross-course projections.
course_rad = np.radians(model_data['cog_deg_clean'].fillna(0))
# Project wind velocity onto the direction of vessel travel.
model_data['wind_along_ms'] = model_data['era5_u10'] * np.sin(course_rad) + model_data['era5_v10'] * np.cos(course_rad)
# Project wind velocity onto the starboard direction across the vessel's course.
model_data['wind_cross_ms'] = model_data['era5_u10'] * np.cos(course_rad) - model_data['era5_v10'] * np.sin(course_rad)
# Project current velocity onto the direction of vessel travel.
model_data['current_along_ms'] = model_data['phy_uo'] * np.sin(course_rad) + model_data['phy_vo'] * np.cos(course_rad)
# Project current velocity onto the starboard direction across the vessel's course.
model_data['current_cross_ms'] = model_data['phy_uo'] * np.cos(course_rad) - model_data['phy_vo'] * np.sin(course_rad)
# Calculate wind speed from its eastward and northward vector components.
model_data['wind_speed_ms'] = np.hypot(model_data['era5_u10'], model_data['era5_v10'])
# Calculate current speed from its eastward and northward vector components.
model_data['current_speed_ms'] = np.hypot(model_data['phy_uo'], model_data['phy_vo'])
# Create coarse location categories so the baseline can account for broad corridor differences.
model_data['latitude_cell'] = np.floor(model_data['Latitude'] / 0.25).astype('int32').astype('string')
model_data['longitude_cell'] = np.floor(model_data['Longitude'] / 0.25).astype('int32').astype('string')
# Add hour and month categories to account for repeated daily and seasonal patterns.
model_data['hour_of_day'] = model_data['Date_utc'].dt.hour.astype('string')
model_data['month'] = model_data['Date_utc'].dt.month.astype('string')
# Define the continuous environmental inputs for the expanded regression.
environment_features = ['wind_speed_ms', 'wind_along_ms', 'wind_cross_ms', 'current_speed_ms', 'current_along_ms', 'current_cross_ms', 'wave_VHM0_m']
# Require usable speed, course, traffic, and environmental measurements.
required_model_fields = ['sog_knots_clean', 'cog_deg_clean', 'log_other_vessels'] + environment_features
# Build the complete-case population using the time, distance, and underway rules.
analysis_all = model_data.loc[model_data['all_time_matches'] & model_data['within_20_km'] & model_data[required_model_fields].notna().all(axis=1) & model_data['sog_knots_clean'].gt(0.5)].copy()
# Save exact high-speed, fixed-position candidate reports for transparent review.
review_excluded = analysis_all.loc[analysis_all['high_sog_stationary_review_flag']].copy()
review_excluded.to_csv(PROCESSED / 'model_excluded_speed_review_candidates.csv', index=False)
# Exclude only those flagged reports from the primary fit; retain other fast craft observations.
analysis = analysis_all.loc[~analysis_all['high_sog_stationary_review_flag']].copy()
# Count remaining measured speeds above 40 knots without clipping them.
high_speed_count = int(analysis['sog_knots_clean'].gt(40).sum())
# Report both the review exclusion and the final modeling population.
print(f'Complete-case moving reports before the targeted review exclusion: {len(analysis_all):,}')
print(f'Flagged reports excluded from the primary fit: {len(review_excluded):,} across {review_excluded["Mmsi"].nunique():,} vessels')
print(f'Model sample: {len(analysis):,} moving reports from {analysis["Mmsi"].nunique():,} vessels')
print(f'Other reports above 40 knots retained for the main fit: {high_speed_count:,}')
# Show the outcome distribution and selected context fields.
display(analysis[['sog_knots_clean', 'wind_speed_ms', 'current_speed_ms', 'wave_VHM0_m', 'other_vessels_cell_hour']].describe(percentiles=[.5, .9, .95, .99]).round(2))


Complete-case moving reports before the targeted review exclusion: 18,111
Flagged reports excluded from the primary fit: 1 across 1 vessels
Model sample: 18,110 moving reports from 8,621 vessels
Other reports above 40 knots retained for the main fit: 0


,sog_knots_clean,wind_speed_ms,current_speed_ms,wave_VHM0_m,other_vessels_cell_hour
count,18110.00,18110.00,18110.00,18110.00,18110.00
mean,8.10,5.92,0.12,1.06,10.56
std,4.74,2.83,0.10,0.76,24.85
min,0.60,0.03,0.00,0.03,0.00
50%,7.50,5.67,0.10,0.86,2.00
90%,14.20,9.70,0.23,2.13,32.00
95%,16.40,11.17,0.30,2.59,62.00
99%,21.00,13.49,0.51,3.60,111.00
max,40.00,18.44,1.15,6.76,421.00


## 5. Compare a context-only baseline with the environmental model

Both models use Ridge regression and the same held-out vessel splits. The baseline includes vessel type, coarse location, hour, month, and local traffic. The expanded model adds wind, current, and wave information. Numeric inputs are standardized within each training pipeline; category encoding is also fitted on training rows only. We report mean absolute error, root mean squared error, and R² across five group-based splits.


A separate movement-quality screen found 16 reports from four vessels with measured speeds above 40 knots while their reported positions stayed within 100 m across multiple days. These reports remain in the cleaned AIS file and are saved for review; only these exact reports are left out of the primary model fit. Other high-speed observations remain eligible because they may represent fast craft.

In [6]:
# Import model pipelines and grouped split tools.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
# Define baseline categories that describe vessel group, broad place, and time.
base_categorical = ['vessel_type_model', 'latitude_cell', 'longitude_cell', 'hour_of_day', 'month']
# Define the local traffic measure for both model versions.
base_numeric = ['log_other_vessels']
# Add environmental measurements only to the expanded model.
expanded_numeric = base_numeric + environment_features
# Create preprocessing steps that fill numeric gaps and standardize numeric features.
def build_model(numeric_features):
    # Scale numeric features and encode categories while ignoring unseen test categories.
    preprocessing = ColumnTransformer([
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), base_categorical),
    ])
    # Fit Ridge regression after preprocessing; regularization stabilizes correlated inputs.
    return Pipeline([('preprocess', preprocessing), ('regression', Ridge(alpha=10.0))])
# Define input columns shared by all model versions.
base_features = base_numeric + base_categorical
# Add the metocean fields to form the expanded feature set.
expanded_features = expanded_numeric + base_categorical
# Create five repeatable train/test splits that keep every MMSI in a single side.
splitter = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
# Store each split's metrics and one set of held-out predictions for review.
metric_rows = []
first_split_predictions = None
# Fit and score both models on identical unseen-vessel groups.
for split_number, (train_index, test_index) in enumerate(splitter.split(analysis, groups=analysis['Mmsi']), start=1):
    # Separate training and test records without splitting the same MMSI across them.
    train = analysis.iloc[train_index]
    test = analysis.iloc[test_index]
    # Fit the context-only model on the training vessels.
    baseline = build_model(base_numeric)
    baseline.fit(train[base_features], train['sog_knots_clean'])
    # Fit the environmental model on exactly the same training vessels.
    expanded = build_model(expanded_numeric)
    expanded.fit(train[expanded_features], train['sog_knots_clean'])
    # Predict speed for vessels not seen during model fitting.
    baseline_prediction = baseline.predict(test[base_features])
    expanded_prediction = expanded.predict(test[expanded_features])
    # Record comparable error metrics for both model versions.
    for model_name, prediction in [('context_only', baseline_prediction), ('context_plus_environment', expanded_prediction)]:
        metric_rows.append({
            'split': split_number,
            'model': model_name,
            'test_rows': len(test),
            'test_vessels': test['Mmsi'].nunique(),
            'MAE_knots': mean_absolute_error(test['sog_knots_clean'], prediction),
            'RMSE_knots': mean_squared_error(test['sog_knots_clean'], prediction) ** 0.5,
            'R2': r2_score(test['sog_knots_clean'], prediction),
        })
    # Save one held-out comparison with actual values and model errors.
    if split_number == 1:
        first_split_predictions = test[['Mmsi', 'Date_utc', 'vessel_type_model', 'latitude_cell', 'longitude_cell', 'other_vessels_cell_hour', 'sog_knots_clean']].copy()
        first_split_predictions['baseline_prediction_knots'] = baseline_prediction
        first_split_predictions['environment_prediction_knots'] = expanded_prediction
        first_split_predictions['environment_model_error_knots'] = first_split_predictions['sog_knots_clean'] - expanded_prediction
# Display each split before summarizing average performance.
metrics = pd.DataFrame(metric_rows)
display(metrics.round(3))
# Summarize average and variation across the five unseen-vessel splits.
metric_summary = metrics.groupby('model')[['MAE_knots', 'RMSE_knots', 'R2']].agg(['mean', 'std']).round(3)
display(metric_summary)
# Calculate the relative MAE change to show whether environment features helped.
mean_mae = metrics.groupby('model')['MAE_knots'].mean()
mae_change_pct = 100 * (mean_mae['context_only'] - mean_mae['context_plus_environment']) / mean_mae['context_only']
print(f'Mean MAE reduction from adding environment: {mae_change_pct:.2f}%')


,split,model,test_rows,test_vessels,MAE_knots,RMSE_knots,R2
0,1,context_only,3633,1725,2.602,3.847,0.388
1,1,context_plus_environment,3633,1725,2.585,3.830,0.393
2,2,context_only,3656,1725,2.624,3.745,0.397
3,2,context_plus_environment,3656,1725,2.606,3.730,0.402
4,3,context_only,3581,1725,2.640,3.723,0.386
5,3,context_plus_environment,3581,1725,2.620,3.705,0.392
6,4,context_only,3778,1725,2.558,3.613,0.407
7,4,context_plus_environment,3778,1725,2.545,3.600,0.411
8,5,context_only,3764,1725,2.593,3.711,0.353
9,5,context_plus_environment,3764,1725,2.582,3.703,0.355


MAE_knots        RMSE_knots            R2       
                              mean    std       mean    std   mean    std
model                                                                    
context_only                 2.603  0.031      3.728  0.084  0.386  0.020
context_plus_environment     2.588  0.029      3.714  0.082  0.391  0.021

Mean MAE reduction from adding environment: 0.59%


'## 6. Check high-speed sensitivity and review candidates

The track screen flags reports above 40 knots for inspection, not automatic deletion. The targeted fixed-position review list is used to remove only exact flagged reports from the primary fit; 1 such report occurs in this matched sample. No remaining report in the modeling sample exceeds 40 knots. The notebook compares test errors with and without reports above 40 knots where any are present, then lists the largest prediction errors for analyst review. A large model error does not identify the reason for a vessel's speed.

In [7]:
# Show how many held-out test reports exceed the earlier 40-knot review threshold.
high_test = first_split_predictions['sog_knots_clean'].gt(40)
print(f'First held-out split reports above 40 knots: {int(high_test.sum()):,}')
# Compare model errors on all test reports with errors after excluding only >40-knot review candidates.
ordinary_test = first_split_predictions.loc[~high_test]
sensitivity_metrics = []
for prediction_column, model_name in [('baseline_prediction_knots', 'context_only'), ('environment_prediction_knots', 'context_plus_environment')]:
    # Calculate full-test errors including high-speed candidates.
    sensitivity_metrics.append({'population': 'all test reports', 'model': model_name, 'n': len(first_split_predictions), 'MAE_knots': mean_absolute_error(first_split_predictions['sog_knots_clean'], first_split_predictions[prediction_column]), 'RMSE_knots': mean_squared_error(first_split_predictions['sog_knots_clean'], first_split_predictions[prediction_column]) ** 0.5})
    # Calculate ordinary-speed errors while leaving training and source data unchanged.
    sensitivity_metrics.append({'population': 'test reports at or below 40 knots', 'model': model_name, 'n': len(ordinary_test), 'MAE_knots': mean_absolute_error(ordinary_test['sog_knots_clean'], ordinary_test[prediction_column]), 'RMSE_knots': mean_squared_error(ordinary_test['sog_knots_clean'], ordinary_test[prediction_column]) ** 0.5})
display(pd.DataFrame(sensitivity_metrics).round(3))
# Rank the largest absolute residuals from the first unseen-vessel split for analyst review.
first_split_predictions['absolute_error_knots'] = first_split_predictions['environment_model_error_knots'].abs()
review_candidates = first_split_predictions.nlargest(20, 'absolute_error_knots')
# Display candidate vessel/time segments without assigning a cause or performance label.
display(review_candidates[['Mmsi', 'Date_utc', 'vessel_type_model', 'latitude_cell', 'longitude_cell', 'other_vessels_cell_hour', 'sog_knots_clean', 'environment_prediction_knots', 'environment_model_error_knots', 'absolute_error_knots']])
# Save model metrics and first-split predictions as reproducible local artifacts.
metrics.to_csv(PROCESSED / 'model_split_metrics.csv', index=False)
first_split_predictions.to_parquet(PROCESSED / 'held_out_speed_predictions.parquet', index=False)
review_candidates.to_csv(PROCESSED / 'speed_review_candidates.csv', index=False)
# Confirm the locations of the derived reports.
print('Saved model metrics, held-out predictions, and review candidates under data/processed/.')


Saved row-level preview omitted from the public copy to avoid republishing individual source records. Run this cell locally with the source files to inspect the rows.


'## 7. How to interpret the result

A lower test MAE means the model's speed estimates are closer on average to held-out vessel reports. Compare the size of any improvement with the split-to-split variation. If the environmental model is only marginally better or varies substantially by split, the practical conclusion is that these environmental matches add limited value for this sample. If it improves consistently, that supports using weather/ocean context to prioritize retrospective review. Neither result proves cause or fuel savings.

The evaluation keeps complete MMSIs in either training or test data, but all splits draw from the same quarter and study area. The result should be checked on a later period or another area before considering wider use.